# Market Basket Analysis: Perilaku Pembelian Pelanggan Online Retail

Analisis ini memakai algoritma **Apriori** untuk mencari produk yang sering dibeli bersamaan, sebagai dasar rekomendasi *product placement*, *bundling*, dan pengelolaan stok.

**Data:** dataset transaksi Online Retail UK, sama dengan yang dipakai pada proyek *Analisis Retensi Pelanggan*, dengan 8 kolom: `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`. Nama kolom asli dipakai apa adanya di seluruh notebook.

**Catatan membaca hasil:** Apriori dijalankan hanya pada **transaksi multi-produk** (transaksi dengan lebih dari 1 produk unik). Karena itu *support*, *confidence*, dan *lift* di bawah dihitung terhadap kelompok transaksi tersebut, bukan terhadap seluruh transaksi.

## 1. Import library dan memuat data

In [1]:
import warnings
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

MIN_SUPPORT = 0.01      # kombinasi produk harus ada di minimal 1% transaksi multi-produk
MIN_CONFIDENCE = 0.70   # aturan harus benar minimal 70% dari kemunculan produk pertama

In [4]:
df = pd.read_excel('Online Retail.xlsx', dtype=str, sheet_name='Online Retail')

kolom_wajib = ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']
kurang = [c for c in kolom_wajib if c not in df.columns]
assert not kurang, f'Kolom berikut tidak ditemukan: {kurang}. Kolom yang ada: {list(df.columns)}'

print('Baris          :', f'{len(df):,}')
print('Invoice unik   :', f"{df['InvoiceNo'].nunique():,}")
print('StockCode unik :', f"{df['StockCode'].nunique():,}")
df.head()

Baris          : 541,909
Invoice unik   : 25,900
StockCode unik : 4,070


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## 2. Pembersihan data

Setiap langkah dicatat jumlah baris dan jumlah invoice-nya (`cleaning_log`) agar angka di README dan CV bisa ditelusuri.

Basket dibentuk dari `InvoiceNo` (satu transaksi) dan `Description` (satu produk). `Quantity` dan `UnitPrice` hanya dipakai untuk membuang baris yang bukan pembelian. Apriori hanya melihat **ada atau tidaknya** produk di sebuah transaksi, bukan berapa banyak yang dibeli, sehingga pembuangan outlier jumlah beli tidak dilakukan (alasannya ada di catatan setelah tabel log).

In [5]:
df_clean = df.copy()
cleaning_log = [('Data mentah', len(df_clean), df_clean['InvoiceNo'].nunique())]

def log_step(nama_langkah):
    cleaning_log.append((nama_langkah, len(df_clean), df_clean['InvoiceNo'].nunique()))

# 1. Hapus baris duplikat (seluruh kolom sama).
#    Jika datanya sama dengan proyek Online Retail UK, jumlahnya seharusnya 5.268 baris.
df_clean = df_clean.drop_duplicates().copy()
log_step('Hapus baris duplikat')

# 2. Ubah Quantity dan UnitPrice menjadi angka; baris yang tidak bisa diubah
#    (atau tanpa InvoiceNo/StockCode) dibuang.
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean['UnitPrice'] = pd.to_numeric(df_clean['UnitPrice'], errors='coerce')
df_clean = df_clean.dropna(subset=['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice']).copy()
log_step('Hapus baris dengan InvoiceNo, StockCode, Quantity, atau UnitPrice tidak valid')

# Standarisasi teks. CustomerID yang kosong TIDAK dibuang: analisis basket memakai nomor invoice,
# bukan pelanggan (sama dengan proyek Online Retail UK yang mempertahankan baris tanpa CustomerID).
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].str.strip()
df_clean['StockCode'] = df_clean['StockCode'].str.strip().str.upper()   # mis. 85123a menjadi 85123A
df_clean['Description'] = df_clean['Description'].str.strip().str.lower()

# 3. Hapus transaksi pembatalan: InvoiceNo diawali huruf 'C'
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')].copy()
log_step('Hapus transaksi pembatalan (InvoiceNo diawali C)')

# 4. Hapus Quantity <= 0 dan UnitPrice <= 0 (koreksi stok atau salah input, bukan pembelian)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)].copy()
log_step('Hapus Quantity <= 0 dan UnitPrice <= 0')

# 5. Hapus kode non-produk (ongkir, diskon, biaya bank, data uji, dan sejenisnya).
#    Kode produk asli berbentuk 5 angka, kadang diikuti huruf (mis. 85123A).
#    Kode seperti POST, DOT, M, D, BANK CHARGES, AMAZONFEE, TEST001 bukan produk dan akan
#    muncul sebagai pasangan palsu pada aturan asosiasi.
#    Konsekuensi: kode produk yang tidak berpola 5 angka ikut terbuang (jumlahnya kecil).
kode_produk = df_clean['StockCode'].str.match(r'^\d{5}[A-Z]*$')
df_clean = df_clean[kode_produk].copy()
log_step('Hapus kode non-produk (POST, DOT, M, D, TEST, dll.)')

# 6. Hapus baris tanpa Description
df_clean = df_clean[df_clean['Description'].notna() & (df_clean['Description'] != '')].copy()
log_step('Hapus baris tanpa Description')

# 7. Satu StockCode bisa punya beberapa Description.
#    Seragamkan memakai nama yang paling sering muncul (dihitung dari jumlah invoice unik).
nama_terbanyak = (
    df_clean.groupby(['StockCode', 'Description'])['InvoiceNo'].nunique()
    .reset_index(name='order_cnt')
    .sort_values(['StockCode', 'order_cnt'], ascending=[True, False], kind='mergesort')
    .drop_duplicates('StockCode')
    .set_index('StockCode')['Description']
)
df_clean['Description'] = df_clean['StockCode'].map(nama_terbanyak)
df_clean = df_clean.reset_index(drop=True)

df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 522504 entries, 0 to 522503
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    522504 non-null  str    
 1   StockCode    522504 non-null  str    
 2   Description  522504 non-null  str    
 3   Quantity     522504 non-null  int64  
 4   InvoiceDate  522504 non-null  str    
 5   UnitPrice    522504 non-null  float64
 6   CustomerID   391150 non-null  str    
 7   Country      522504 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 69.0 MB


In [15]:
log_df = pd.DataFrame(cleaning_log, columns=['langkah', 'baris_tersisa', 'invoice_tersisa'])
log_df['baris_dibuang'] = (-log_df['baris_tersisa'].diff()).fillna(0).astype(int)
log_df['invoice_dibuang'] = (-log_df['invoice_tersisa'].diff()).fillna(0).astype(int)
log_df

,langkah,baris_tersisa,invoice_tersisa,baris_dibuang,invoice_dibuang
0,Data mentah,541909,25900,0,0
1,Hapus baris duplikat,536641,25900,5268,0
2,"Hapus baris dengan InvoiceNo, StockCode, Quant...",536641,25900,0,0
3,Hapus transaksi pembatalan (InvoiceNo diawali C),527390,22064,9251,3836
4,Hapus Quantity <= 0 dan UnitPrice <= 0,524878,19960,2512,2104
5,"Hapus kode non-produk (POST, DOT, M, D, TEST, ...",522504,19773,2374,187
6,Hapus baris tanpa Description,522504,19773,0,0


**Catatan: kenapa tidak ada langkah pembuangan outlier.** Pembuangan outlier (misalnya Z-score pada `Quantity`) cocok untuk analisis yang memakai jumlah atau nilai, tetapi tidak untuk Apriori. Apriori hanya bertanya "produk ini ada di transaksi ini atau tidak", jadi baris berjumlah besar tetap menyumbang informasi yang sah (produk itu memang dibeli bersama produk lain). Membuang baris tersebut justru bisa menghilangkan produk dari sebuah transaksi dan mengubah transaksi multi-produk menjadi satu produk, sehingga hasilnya bergeser tanpa alasan yang jelas.

## 3. Menyiapkan data basket

Setiap baris adalah satu transaksi (`InvoiceNo`), setiap kolom adalah satu produk (`Description`), dan nilainya `True` bila produk itu ada di transaksi tersebut.

In [7]:
# Jumlah produk unik di setiap transaksi
produk_per_invoice = df_clean.groupby('InvoiceNo')['Description'].nunique()
invoice_multi = produk_per_invoice[produk_per_invoice > 1].index
n_multi = len(invoice_multi)

print('Transaksi seluruhnya      :', f'{len(produk_per_invoice):,}')
print('Transaksi multi-produk    :', f'{n_multi:,}')
print('Persentase multi-produk   : {:.1%}'.format(n_multi / len(produk_per_invoice)))
print('Produk unik (seluruhnya)  :', f"{df_clean['Description'].nunique():,}")

Transaksi seluruhnya      : 19,773
Transaksi multi-produk    : 18,273
Persentase multi-produk   : 92.4%
Produk unik (seluruhnya)  : 3,765


### Matriks basket untuk transaksi multi-produk

Support dihitung terhadap seluruh **transaksi multi-produk**. Produk yang muncul di kurang dari `MIN_SUPPORT` transaksi tersebut dibuang dari matriks, karena tidak mungkin berada di kombinasi yang lolos. Hasil Apriori tetap sama persis, tetapi matriksnya jauh lebih kecil sehingga lebih cepat dan hemat memori. Transaksi tetap dihitung semuanya (`reindex`), supaya penyebut support tidak berubah.

In [8]:
df_multi = df_clean[df_clean['InvoiceNo'].isin(invoice_multi)]

# Hanya simpan produk yang muncul di minimal MIN_SUPPORT dari transaksi multi-produk
invoice_per_produk = df_multi.groupby('Description')['InvoiceNo'].nunique()
produk_sering = invoice_per_produk[invoice_per_produk / n_multi >= MIN_SUPPORT].index
df_multi = df_multi[df_multi['Description'].isin(produk_sering)]

# Matriks transaksi x produk (True/False). reindex menjaga semua transaksi multi-produk tetap ada,
# termasuk yang tidak memuat produk sering sama sekali.
basket_filter = (
    (pd.crosstab(df_multi['InvoiceNo'], df_multi['Description']) > 0)
    .reindex(invoice_multi, fill_value=False)
)

assert len(basket_filter) == n_multi
print('Ukuran matriks basket     :', basket_filter.shape, '(transaksi x produk)')
print('Produk dipertahankan      :', f'{len(produk_sering):,} dari {len(invoice_per_produk):,}')

Ukuran matriks basket     : (18273, 898) (transaksi x produk)
Produk dipertahankan      : 898 dari 3,760


## 4. Menerapkan algoritma Apriori

Parameter: *minimum support* 1% dan *minimum confidence* 70% (diatur di `MIN_SUPPORT` dan `MIN_CONFIDENCE` pada bagian 1). Support dihitung terhadap **transaksi multi-produk** (`basket_filter`).

In [10]:
frequent_itemset = (
    apriori(basket_filter, min_support=MIN_SUPPORT, use_colnames=True, low_memory=True)
    .sort_values('support', ascending=False)
    .reset_index(drop=True)
)
frequent_itemset['product_cnt'] = frequent_itemset['itemsets'].apply(len)
frequent_itemset

,support,itemsets,product_cnt
0,0.122914,frozenset({white hanging heart t-light holder}),1
1,0.113446,frozenset({jumbo bag red retrospot}),1
2,0.107536,frozenset({regency cakestand 3 tier}),1
3,0.091392,frozenset({party bunting}),1
4,0.085591,frozenset({lunch bag red retrospot}),1
...,...,...,...
2520,0.010015,"frozenset({jumbo bag pink vintage paisley, lun...",2
2521,0.010015,"frozenset({herb marker rosemary, herb marker c...",3
2522,0.010015,"frozenset({herb marker mint, herb marker chive...",3
2523,0.010015,"frozenset({herb marker thyme, herb marker chiv...",3


### Hitung support, confidence, dan lift untuk setiap aturan

- **Support**: seberapa sering kombinasi produk muncul di transaksi multi-produk.
- **Confidence**: dari transaksi yang memuat produk A, berapa persen yang juga memuat produk B.
- **Lift**: seberapa sering A dan B dibeli bersamaan dibanding jika keduanya dibeli secara saling lepas (independen). Nilai di atas 1 berarti ada asosiasi positif.

In [11]:
# Versi mlxtend terbaru meminta parameter num_itemsets, versi lama tidak.
try:
    rules = association_rules(frequent_itemset, num_itemsets=n_multi,
                              metric='confidence', min_threshold=MIN_CONFIDENCE)
except TypeError:
    rules = association_rules(frequent_itemset, metric='confidence', min_threshold=MIN_CONFIDENCE)

assert len(rules) > 0, 'Tidak ada aturan yang lolos. Coba turunkan MIN_SUPPORT atau MIN_CONFIDENCE.'

product_association = (
    rules.sort_values(['support', 'confidence'], ascending=[False, False])
    .reset_index(drop=True)
)
print('Jumlah aturan asosiasi:', len(product_association))
product_association

Jumlah aturan asosiasi: 467


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({green regency teacup and saucer}),frozenset({roses regency teacup and saucer}),0.055437,0.058119,0.041974,0.757157,13.027805,1.0,0.038753,3.878561,0.977427,0.586391,0.742172,0.739690
1,frozenset({roses regency teacup and saucer}),frozenset({green regency teacup and saucer}),0.058119,0.055437,0.041974,0.722222,13.027805,1.0,0.038753,3.400427,0.980209,0.586391,0.705919,0.739690
2,frozenset({pink regency teacup and saucer}),frozenset({green regency teacup and saucer}),0.041591,0.055437,0.034587,0.831579,15.000436,1.0,0.032281,5.608343,0.973839,0.553900,0.821694,0.727734
3,frozenset({pink regency teacup and saucer}),frozenset({roses regency teacup and saucer}),0.041591,0.058119,0.032726,0.786842,13.538574,1.0,0.030309,4.418703,0.966328,0.488562,0.773689,0.674965
4,frozenset({gardeners kneeling pad cup of tea}),frozenset({gardeners kneeling pad keep calm}),0.041427,0.049910,0.029880,0.721268,14.451462,1.0,0.027813,3.408618,0.971030,0.486198,0.706626,0.659976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,frozenset({herb marker parsley}),"frozenset({herb marker mint, herb marker chives})",0.013025,0.010398,0.010015,0.768908,73.948673,1.0,0.009879,4.282278,0.999495,0.746939,0.766479,0.866033
463,frozenset({herb marker parsley}),"frozenset({herb marker thyme, herb marker chiv...",0.013025,0.010343,0.010015,0.768908,74.339936,1.0,0.009880,4.282515,0.999567,0.750000,0.766492,0.868581
464,frozenset({herb marker rosemary}),"frozenset({herb marker chives, herb marker par...",0.013134,0.010562,0.010015,0.762500,72.192552,1.0,0.009876,4.166055,0.999273,0.732000,0.759965,0.855343
465,frozenset({herb marker mint}),"frozenset({herb marker chives, herb marker par...",0.013134,0.010562,0.010015,0.762500,72.192552,1.0,0.009876,4.166055,0.999273,0.732000,0.759965,0.855343


## 5. Aturan dengan lift tertinggi

Aturan A -> B dan B -> A sering sama-sama lolos dan punya lift yang sama. Supaya daftar ini tidak berisi pasangan yang sama dua kali, ditampilkan **satu aturan per kombinasi produk** (yang confidence-nya paling tinggi).

In [12]:
rules_unik = (
    product_association
    .assign(kombinasi=product_association.apply(
        lambda r: frozenset(r['antecedents'] | r['consequents']), axis=1))
    .sort_values(['lift', 'confidence'], ascending=[False, False])
    .drop_duplicates('kombinasi')
    .reset_index(drop=True)
)

top_lift = rules_unik.head(10)
top_lift[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
0,"frozenset({herb marker thyme, herb marker mint})","frozenset({herb marker rosemary, herb marker p...",0.010179,0.902913,76.739174
1,frozenset({herb marker chives}),"frozenset({herb marker mint, herb marker parsl...",0.010015,0.875598,75.470773
2,"frozenset({herb marker thyme, herb marker basil})","frozenset({herb marker rosemary, herb marker p...",0.010179,0.885714,75.277475
3,frozenset({herb marker chives}),"frozenset({herb marker thyme, herb marker pars...",0.010015,0.875598,75.116450
4,frozenset({herb marker chives}),"frozenset({herb marker rosemary, herb marker p...",0.010015,0.875598,74.417692
5,"frozenset({herb marker rosemary, herb marker p...",frozenset({herb marker thyme}),0.011109,0.944186,73.106405
6,"frozenset({herb marker thyme, herb marker mint})",frozenset({herb marker rosemary}),0.010781,0.956311,72.811104
7,"frozenset({herb marker thyme, herb marker basil})",frozenset({herb marker rosemary}),0.010945,0.952381,72.511905
8,"frozenset({herb marker thyme, herb marker mint})",frozenset({herb marker parsley}),0.010617,0.941748,72.304846
9,frozenset({herb marker thyme}),frozenset({herb marker rosemary}),0.012094,0.936441,71.298252


## 6. Kesimpulan

In [13]:
# Ringkasan dihitung langsung dari hasil, supaya angka di kesimpulan selalu sesuai data.
r1 = product_association.iloc[0]   # aturan dengan support tertinggi
r2 = rules_unik.iloc[0]            # aturan dengan lift tertinggi

def nama(itemset):
    return ', '.join(sorted(itemset))

print(f"Jumlah aturan asosiasi: {len(product_association)} "
      f"({len(rules_unik)} kombinasi produk unik), dari {n_multi:,} transaksi multi-produk")
print()
print("Aturan dengan support tertinggi:")
print(f"  {nama(r1['antecedents'])}  ->  {nama(r1['consequents'])}")
print(f"  - Produk pertama muncul di {r1['antecedent support']:.1%} transaksi multi-produk")
print(f"  - Support {r1['support']:.1%}, confidence {r1['confidence']:.1%}, lift {r1['lift']:.2f}")
print(f"  - Kedua produk dibeli bersamaan {r1['lift']:.1f}x lebih sering daripada jika dibeli secara saling lepas")
print()
print("Aturan dengan lift tertinggi:")
print(f"  {nama(r2['antecedents'])}  ->  {nama(r2['consequents'])}")
print(f"  - Support {r2['support']:.1%}, confidence {r2['confidence']:.1%}, lift {r2['lift']:.2f}")

Jumlah aturan asosiasi: 467 (282 kombinasi produk unik), dari 18,273 transaksi multi-produk

Aturan dengan support tertinggi:
  green regency teacup and saucer  ->  roses regency teacup and saucer
  - Produk pertama muncul di 5.5% transaksi multi-produk
  - Support 4.2%, confidence 75.7%, lift 13.03
  - Kedua produk dibeli bersamaan 13.0x lebih sering daripada jika dibeli secara saling lepas

Aturan dengan lift tertinggi:
  herb marker mint, herb marker thyme  ->  herb marker parsley, herb marker rosemary
  - Support 1.0%, confidence 90.3%, lift 76.74


### Rekomendasi

Dua aturan teratas sama-sama berasal dari **produk satu seri**: cangkir dan lepek Regency (hijau -> mawar, confidence 75.7%, lift 13.0) serta empat herb marker (mint, thyme -> parsley, rosemary, confidence 90.3%, lift 76.7). Pembeli satu varian cenderung membeli varian lain dari seri yang sama.

- **Jual set lengkap:** tawarkan paket seri (misalnya satu set herb marker) di samping penjualan satuan, dan tampilkan varian seri yang sama berdampingan di halaman produk atau di rak.
- **Jaga stok seimbang antar varian:** saat satu varian dipromosikan atau habis, pantau varian pasangannya, karena keduanya sering dibeli bersamaan.
- **Langkah berikutnya:** aturan satu seri mudah ditebak, jadi nilai tambahnya kecil. Telusuri tabel `product_association` untuk pasangan lintas seri atau kategori, karena di situ fitur "sering dibeli bersamaan" paling berguna.

### Batasan

- Aturan asosiasi menunjukkan produk yang *sering dibeli bersamaan*, bukan bukti bahwa bundling atau penataan baru akan menaikkan penjualan. Rekomendasi di atas sebaiknya diuji (misalnya A/B test) sebelum diterapkan penuh.
- Analisis hanya mencakup transaksi multi-produk, sehingga hasilnya tidak menggambarkan pelanggan yang membeli satu jenis produk saja.
- Aturan dengan lift sangat tinggi sering berasal dari produk satu seri atau varian (misalnya beda warna) yang memang dijual berpasangan. Ini wajar, tetapi bukan temuan yang mengejutkan. Untuk bundling, pasangan lintas kategori biasanya lebih bernilai.
- Produk dengan nama yang sama tetapi kode berbeda dianggap satu produk, karena basket dibentuk dari `Description`.